# ADNI

## INIT

In [1]:
from data_model.DataCleaner import DataCleaner, update_variables_support_file
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [17]:
file_codes = ['ADNIMERGE', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 
               'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T']
    
#'ADNIMERGE', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 
#                'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T', 
#                'UCSDVOL', 'UPENN_ROI_MARS']         'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS']

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES',

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'
# 'UCSDVOL', 'UPENN_ROI_MARS',  --> do NOT use FreeSurfer but other model or Atlas so not comparable

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [18]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI', 'custom.file_code' : file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())
print(len(zip_files.keys()))


KeyboardInterrupt: 

# Support file managment
operazione per popolare il file support file per i file considerati

In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

In [ ]:
for file_name in list(zip_files.keys()):
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    print(file_code)
    if file_code not in list(support_file['file_code']):
        print('not found in excel')
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

In [ ]:
new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)

## IF SUPPORT FILE already populated

In [19]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)


The ADNI_variables_cleaned1 file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned1 file has restored the previous information of the file_code: ['ADNIMERGE', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T']
Open the file and verify it, if needed update the variables names and metadata


In [2]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'

In [ ]:
new_support_file_name = 'ADNI_variables_cleaned1'

Open the new_support_file and fill in the new variable codes.

# FILE SPECIFIC DATA CLEANING 1


## Abeta & Tau in CSF - Elecsys

### UPENNBIOMK_ROCHE_ELECSYS

In [ ]:
file_code = 'UPENNBIOMK_ROCHE_ELECSYS'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'TAU', 'PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'elecsys'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")



In [ ]:
final_df.head()

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'TTAU_CSF', 'PT181_CSF', 'AB4240_CSF', 'TTAU_AB42_CSF', 'PT181_AB42_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELECSYS")
print(percentile_df.dropna(how='all'))

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENNBIOMK_ADNIDIAN_ES_2017

In [ ]:
file_code = 'UPENNBIOMK_ADNIDIAN_ES_2017'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA','TAU','PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'elecsys'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")



In [ ]:
final_df.head()

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'AB4240_CSF', 'TTAU_CSF', 'PT181_CSF', 'TTAU_AB42_CSF', 'PT181_AB42_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELECSYS")
print(percentile_df.dropna(how='all'))

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Abeta Tau altri metodi

### EUROIMMUN

In [ ]:
file_code = 'EUROIMMUN'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['BETA_AMYLOID_1_40', 'BETA_AMYLOID_1_42']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['EXAMDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'elisa'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")

In [ ]:
final_df

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'AB4240_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELISA")
print(percentile_df.dropna(how='all'))

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### FUJIREBIOABETA

In [ ]:
file_code = 'FUJIREBIOABETA'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'ABETA40']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['EXAMDATE'])

In [ ]:
no_none_df

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'lumipulse'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")


In [ ]:
final_df

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'AB4240_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo LUMIPULSE")
print(percentile_df.dropna(how='all'))

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### SALADAX_BIOMEDICAL

In [ ]:
file_code = 'SALADAX_BIOMEDICAL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'TOTALTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['EXAMDATE'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'saladax'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")


In [ ]:
final_df

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
var_of_interest = ['AB42_CSF', 'TTAU_CSF', 'TTAU_AB42_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo SALADAX")
print(percentile_df.dropna(how='all'))

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### MESOSCALE

In [ ]:
file_code = 'MESOSCALE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA40', 'ABETA42', 'TAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA42'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDTE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDTE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'mesoscalediscovery'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')  

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df.head()

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'TTAU_CSF', 'AB4240_CSF', 'TTAU_AB42_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo MESOSCALEDISCOVERY")
print(percentile_df.dropna(how='all'))

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = 'ADNI_MESOSCALE_23Oct2025_01.csv'

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENNBIOMK_MASTER
tenere solo parametri ribilanciati e non quelli raw
filtrare per tenere per ciascuna visita solo la riga con BATCH == MEDIAN altrimenti calcolare np.median tra i valori, se una sola visita ==> prendere quel valore e basta

In [ ]:
file_code = 'UPENNBIOMK_MASTER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA', 'PTAU', 'TAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA'])

In [ ]:
reduced_df = dataCleaner.get_mean_row_per_visit(no_none_df)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(reduced_df, ['DRAWDTE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDTE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'alzbio3'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df.head()

In [ ]:
var_of_interest = ['AB42_CSF', 'PT181_CSF', 'TTAU_CSF', 'PT181_AB42_CSF', 'TTAU_AB42_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ALZBIO3")
print(percentile_df.dropna(how='all'))

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENN_2DUPLC_CRM

In [ ]:
file_code = 'UPENN_2DUPLC_CRM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA40', 'ABETA42']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA42'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
datefix_df['METHOD'] = 'massospectrometry'
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df.head()

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'AB4240_CSF']
df_interest = final_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo MASSOSPECTOMETRY")
print(percentile_df.dropna(how='all'))

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Volumi

### UCSF Longitudinal dataset

In [58]:
file_codes = ['UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL']
file_code = file_codes[1]

In [59]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [60]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
#if file_code == 'UCSFFSL': #the other UCSF longitudinal files have just partial immages segmentation
    #no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [61]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [62]:
if file_code == 'UCSFFSL':
    datefix_df['FSVERSION'] = '4.4'
    datefix_df['FLDSTRENG'] = datefix_df['FLDSTRENG'].astype(str) + 'T'
else:
    datefix_df['FSVERSION'] = '5.1'

#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [63]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# Filter for Quality Chek parameters for Segmented immages
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard, se differiscono capire se metterlo post rename

# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df),'\n', qc_col)

before 3311 
after 3123 
 ['OVERALLQC', 'TEMPQC', 'VENTQC', 'LHIPQC', 'RHIPQC']


In [64]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")


Colonna: COHORT: 1
Colonna: RID: 0
Colonna: VISCODE: 0
Colonna: VISIT_MONTH: 0
Colonna: EXAMDATE: 0
Colonna: IMAGEUID: 0
Colonna: STATUS: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 8
Colonna: LFusiform: 8
Colonna: LHippocampus: 0
Colonna: LVentricles: 8
Colonna: LMidTemp: 9
Colonna: REntorhinal: 8
Colonna: RFusiform: 9
Colonna: RHippocampus: 0
Colonna: RVentricles: 8
Colonna: RMidTemp: 8
Colonna: FSVERSION: 0


In [65]:
final_df['FSVERSION'].unique()

array(['5.1'], dtype=object)

In [66]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [67]:
# optaining automatically info to save the file
if file_code == 'UCSFFSL':
    lst_population = ['ADNI1','ADNIGO','ADNI2']
else:
    lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [68]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX 
complete    4485\
partial        1\
hanno solo VISITCODE e non VISITCODE2 inoltre non hanno info sulla popolazione --> da inserire manualmente?

In [79]:
file_code = 'UCSFFSX' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [80]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
#no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [81]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  184
Adopted visit selection strategy:
 VISITCODE priority           187
first row, same VISITCODE    139
Name: count, dtype: int64


In [82]:
datefix_df['FSVERSION'] = '4.3'
datefix_df['FLDSTRENG'] = datefix_df['FLDSTRENG'].astype(str) + 'T'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [83]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# Filter for Quality Chek parameters for Segmented immages
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard, se differiscono capire se metterlo post rename

# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df), '\n', qc_col)

before 4144 
after 4076 
 ['OVERALLQC', 'TEMPQC', 'VENTQC']


In [84]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")

Colonna: RID: 0
Colonna: VISCODE: 137
Colonna: VISIT_MONTH: 0
Colonna: EXAMDATE: 0
Colonna: FLDSTRENG: 0
Colonna: IMAGEUID: 0
Colonna: STATUS: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 15
Colonna: LFusiform: 16
Colonna: LHippocampus: 0
Colonna: LVentricles: 15
Colonna: LMidTemp: 15
Colonna: REntorhinal: 15
Colonna: RFusiform: 15
Colonna: RHippocampus: 0
Colonna: RVentricles: 15
Colonna: RMidTemp: 16
Colonna: FSVERSION: 0


In [85]:
final_df['FSVERSION'].unique()

array(['4.3'], dtype=object)

In [86]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [87]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [88]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFX7

partial     11091\
complete      849

In [89]:
file_code = 'UCSFFSX7' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [90]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
#no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') 

In [91]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  246
Adopted visit selection strategy:
 VISITCODE priority    584
Min null values         2
Name: count, dtype: int64


In [92]:
datefix_df['FIELD_STRENGTH'] = datefix_df['FIELD_STRENGTH'].str.extract('([0-9.]+)', expand=False)+'T'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [93]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# final filter on Quality Check for segmented volumes
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard

# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))
print(qc_col)

before 11339 
after 847
['OVERALLQC', 'TEMPQC', 'VENTQC', 'HIPPOQC']


In [94]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")

Colonna: COHORT: 0
Colonna: RID: 0
Colonna: VISCODE: 1
Colonna: VISIT_MONTH: 0
Colonna: IMAGEUID: 0
Colonna: FLDSTRENG: 0
Colonna: EXAMDATE: 0
Colonna: STATUS: 0
Colonna: FSVERSION: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 120
Colonna: LFusiform: 277
Colonna: LHippocampus: 115
Colonna: LVentricles: 15
Colonna: LMidTemp: 201
Colonna: REntorhinal: 120
Colonna: RFusiform: 277
Colonna: RHippocampus: 115
Colonna: RVentricles: 15
Colonna: RMidTemp: 201


In [97]:
final_df['FSVERSION'].unique()

array(['7.4.1'], dtype=object)

In [98]:
# update the new info support file
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [99]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [100]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX6
complete    2222\
partial       18

In [101]:
file_code = 'UCSFFSX6' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [102]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
#no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [103]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [104]:
datefix_df['FSVERSION'] = '6.0'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [105]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# final filter on Quality Check for segmented volumes
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard

# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))


before 2240 
after 2222


In [106]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")

Colonna: COHORT: 0
Colonna: RID: 0
Colonna: VISCODE: 2
Colonna: VISIT_MONTH: 0
Colonna: IMAGEUID: 0
Colonna: EXAMDATE: 0
Colonna: STATUS: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 275
Colonna: LFusiform: 414
Colonna: LHippocampus: 268
Colonna: LVentricles: 86
Colonna: LMidTemp: 182
Colonna: REntorhinal: 275
Colonna: RFusiform: 415
Colonna: RHippocampus: 268
Colonna: RVentricles: 86
Colonna: RMidTemp: 182
Colonna: FSVERSION: 0


In [107]:
final_df['FSVERSION'].unique()

array(['6.0'], dtype=object)

In [108]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [109]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [110]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX51

In [111]:
file_code = 'UCSFFSX51' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [112]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [113]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  333
Adopted visit selection strategy:
 VISITCODE priority           272
first row, same VISITCODE    265
Min null values                1
Name: count, dtype: int64


In [114]:
datefix_df['FSVERSION'] = '5.1'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [115]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# final filter on Quality Check for segmented volumes
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard


In [116]:
# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))
print(qc_col)

before 4350 
after 4038
['OVERALLQC', 'TEMPQC', 'VENTQC', 'LHIPQC', 'RHIPQC']


In [117]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")

Colonna: COHORT: 0
Colonna: RID: 0
Colonna: VISCODE: 0
Colonna: VISIT_MONTH: 0
Colonna: EXAMDATE: 0
Colonna: IMAGEUID: 0
Colonna: STATUS: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 67
Colonna: LFusiform: 69
Colonna: LHippocampus: 0
Colonna: LVentricles: 67
Colonna: LMidTemp: 68
Colonna: REntorhinal: 67
Colonna: RFusiform: 67
Colonna: RHippocampus: 0
Colonna: RVentricles: 68
Colonna: RMidTemp: 68
Colonna: FSVERSION: 0


In [118]:
final_df['FSVERSION'].unique()

array(['5.1'], dtype=object)

In [119]:

infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [120]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [121]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX51_ADNI1_3T
partial    484


In [122]:
file_code = 'UCSFFSX51_ADNI1_3T' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [123]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [124]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  1
Adopted visit selection strategy:
 VISITCODE priority    1
Name: count, dtype: int64


In [125]:
datefix_df['FSVERSION'] = '5.1'
datefix_df['FLDSTRENG'] = '3T'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION', 'FLDSTRENG'], prefix='raw')   

In [126]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# final filter on Quality Check for segmented volumes
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard

In [127]:
# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))
print(qc_col)

before 483 
after 435
['OVERALLQC', 'TEMPQC', 'VENTQC']


In [128]:
nan_counts = final_df.isna().sum()
for col, count in nan_counts.items():
    print(f"Colonna: {col}: {count}")

Colonna: RID: 0
Colonna: VISCODE: 0
Colonna: VISIT_MONTH: 0
Colonna: EXAMDATE: 0
Colonna: IMAGEUID: 0
Colonna: STATUS: 0
Colonna: ICV: 0
Colonna: LEntorhinal: 14
Colonna: LFusiform: 14
Colonna: LHippocampus: 0
Colonna: LVentricles: 14
Colonna: LMidTemp: 14
Colonna: REntorhinal: 14
Colonna: RFusiform: 14
Colonna: RHippocampus: 0
Colonna: RVentricles: 14
Colonna: RMidTemp: 14
Colonna: FSVERSION: 0
Colonna: FLDSTRENG: 0


In [129]:
final_df['FSVERSION'].unique()

array(['5.1'], dtype=object)

In [130]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [131]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [132]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSDVOL --- NO FreeSurfer

In [133]:
file_code = 'UCSDVOL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [134]:
# Important columns
columns_must_be_verified = ['BRAIN', 'EICV', 'VENTRICLES', 'LHIPPOC', 'RHIPPOC']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# Convert QCPASS values from 1/0 to 'complete'/'partial'
no_none_df = dataCleaner.convert_qcpass_values(no_none_df, col_name='QCPASS')
#no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='QCPASS')

In [135]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [136]:
datefix_df['FSVERSION'] = 'NOT FreeSurfer'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [137]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)
# Filter for Quality Chek parameters for Segmented immages
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard, se differiscono capire se metterlo post rename

# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))
print(qc_col)

NO Quality Check parameters present in the dataframe
before 3128 
after 3128
[]


In [138]:
final_df['FSVERSION'].unique()

array(['NOT FreeSurfer'], dtype=object)

In [139]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [140]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [141]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENN_ROI_MARS --- NO FreeSurfer

In [142]:
file_code = 'UPENN_ROI_MARS' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [143]:
# Important columns
columns_must_be_verified = ['R702', 'R525', 'R517', 'R122', 'R123', 'R116', 'R117', 'R47', 'R48']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['R702']) # se manca ICV non si può normalizzare
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# VERIFICARE ma non da usare ---> se all partial ==> elimina file dai cleaned non ci interessa
# no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [144]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [149]:
datefix_df['FSVERSION'] = 'NOT FreeSurfer'
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'FSVERSION'], prefix='raw')   

In [150]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

# final filter on Quality Check for segmented volumes
final_df = dataCleaner.volume_quality_filter(renamed_df) ## verificare nomi variabili QC standard

NO Quality Check parameters present in the dataframe


In [151]:
# QC parameter are no longer usefull --> drop from final_df and new_support_file
qc_col = [x for x in final_df if 'QC' in x]
final_df = final_df.drop(columns=qc_col)
new_support_file = new_support_file[~((new_support_file['file_code'] == file_code) & (new_support_file['variable_code'].isin(qc_col)))]
print('before', len(renamed_df), '\nafter', len(final_df))


before 840 
after 840


In [152]:
final_df['FSVERSION'].unique()

array(['NOT FreeSurfer'], dtype=object)

In [153]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [154]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

totale subject: 840
subjects with multiple visits:  0


In [155]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [156]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Multiparametre dataset


### ADNI MERGE

In [157]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [158]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [159]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [160]:
processed_df = datefix_df.copy(deep=True)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['AGE_bl', 'VISIT_MONTH'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE
processed_df['FLDSTRENG'] = processed_df['FLDSTRENG'].str.extract('([0-9.]+)', expand=False)+'T'
processed_df['FSVERSION'] = processed_df['FSVERSION'].str.extract('([0-9.]+)', expand=False)

In [161]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [162]:
final_df['FSVERSION'].unique()

array(['4.3', nan, '5.1', '6.0'], dtype=object)

In [163]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [165]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADSP_PHC_BIOMARKER

In [ ]:
file_code = 'ADSP_PHC_BIOMARKER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'Tau_RAW', 'pTau_RAW', 'AB42_RAW', 'AT_class']
single_column_required = ['PHC_Diagnosis']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = dataCleaner.binarization_gender(datefix_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df, new_var = dataCleaner.convert_to_dummies_ATNC_profile(processed_df, col_name='AT_class')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
processed_df['PHC_Race'] = processed_df['PHC_Race'].map(mapping)

In [ ]:
filtered_df =dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'] + new_var, prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)

In [ ]:
renamed_df

In [ ]:
renamed_df['METHOD'] = renamed_df['METHOD'].map({'xMAP': 'alzbio3'})

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")

In [ ]:
final_df.head()

In [ ]:

var_of_interest = ['AB42_CSF',  'PT181_CSF', 'TTAU_CSF', 'PT181_AB42_CSF', 'TTAU_AB42_CSF']
df_interest = AbT_df[var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELECSYS")
print(percentile_df.dropna(how='all'))

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADNI_DIAN_COMPARISON

In [ ]:
file_code = 'ADNI_DIAN_COMPARISON'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [ ]:
# Important columns
# in questo caso non metto altri filtri perche mancano davvero tante info e preferisco tenere un file molto frammentato, altrimenti si potrebbero togliere le righe che non hanno queste info, ma perderemmo informazioni su pop che magari sono riutilizzabili
single_column_required = ['EXAMDATE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
#no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, single_column_required)

In [ ]:
columns_must_be_verified = ['MR_TOTV_INTRACRANIAL', 'MR_TOTV_HIPPOCAMPUS', 'CSF_ELC_AB42', 'CSF_ELC_PTAU', 'CSF_ELC_TAU', 'CSF_ELC_AB40', 'CSF_ELC_AB4240', 'MSP_AB40', 'MSP_AB42']
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)


In [ ]:
# CATHEGORIZATION Gender
col_name = 'GENDER'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name].astype('int64')
# mapp gender in to classes: 0 = 'female' --> 0, 1 = 'male' --> 1 --> non necessario perchè già nel formato corretto

In [ ]:
# CATEGORIZZAZIONE Status Maritale
col_name = 'MARISTAT'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name]
# Map marital status to classes: 1 ='married' -> 1, 6 = 'Living as married' --> 1, 3 = 'divorced' -> 2, 4 = 'separated' --> 2,  2 ='widowed' -> 3, 5 = 'never married' -> 0, 7 = 'Other' --> nan, 9 = 'Unknown' --> nan
mapping = {1: 1, 6: 1, 3: 2, 4: 2, 2: 3, 5: 0, 1.0: 1, 6.0: 1, 3.0: 2, 4.0: 2, 2.0: 3, 5.0: 0}
processed_df[col_name] = processed_df[col_name].map(mapping)

In [ ]:
# CATEGORIZZAZIONE Etnicity
col_name = 'HISPANIC'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name].astype('int64')
# mapp Ethnicity in to classes: 0 = 'not hisp/latino' --> 1, 1 = 'hisp/latino' --> 0
processed_df[col_name] = processed_df[col_name].map(lambda x: 1 if x in [0, 0.0] else (0 if x in [1, 1.0] else np.nan))

In [ ]:
# CATEGORIZZAZIONE Race
col_name = 'RACE'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name].astype('int64')
# Map race to classes: 1 = 'White' --> 5,  2 = 'Black' --> 4, 5 = 'Asian'--> 2, 3 = 'Am Indian/Alaskan' --> 1, 4 = 'Hawaiian/Other PI' -->3)
mapping = {1: 5, 2: 4, 3: 1, 4: 3, 5: 2, 1.0: 5, 2.0: 4, 3.0: 1, 4.0: 3, 5.0: 2}
processed_df[col_name] = processed_df[col_name].map(mapping)

In [ ]:
# Categorizzazione mutazione DIAN 
# 

col_name = 'DIAN_GROUP'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name].astype('int64')
# Map race to classes: 
#0 (ADNI, no DIAN) --> nan
# 1 (no mutation present) --> 0
# 2 (mutation present but cognitive normal) --> 1
# 3 (mutation present andcol_name = 2
mapping = {1: 0, 2: 1, 3: 2}
processed_df[col_name] = processed_df[col_name].map(mapping)

In [ ]:
processed_df = dataCleaner.uniform_APOE_format(df=processed_df, col_name='DIAN_APOE')
processed_df = dataCleaner.APOE_4_count(df=processed_df, col_name='DIAN_APOE')

In [ ]:
processed_df

In [ ]:
methods_df = dataCleaner.handel_same_variable_different_methods(
    df=processed_df, 
    var1=['CSF_ELC_AB42', 'CSF_ELC_PTAU', 'CSF_ELC_TAU', 'CSF_ELC_AB40', 'CSF_ELC_AB4240'], 
    var2=['MSP_AB40', 'MSP_AB42'], 
    method1='elecsys', method2='massospectrometry', 
    mapping={'MSP_AB40':'CSF_ELC_AB40', 'MSP_AB42':'CSF_ELC_AB42'})

In [ ]:

filtered_df = dataCleaner.filter_variables(methods_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH', 'METHOD', 'APOE_4'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
# Now pass the DataFrame to the get_ATN_profile function
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df, "cutoffs.json")
  # This will show the resulting DataFrame

In [ ]:
final_df

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF',  'AB4240_CSF', 'PT181_CSF', 'TTAU_CSF', 'PT181_AB42_CSF', 'TTAU_AB42_CSF']
df_interest = final_df[final_df['METHOD']=='elecsys'][var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELECSYS")
print(percentile_df.dropna(how='all'))

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
   
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
final_df

In [ ]:
var_of_interest = ['AB40_CSF', 'AB42_CSF', 'TTAU_CSF', 'PT181_CSF', 'AB4240_CSF', 'TTAU_AB42_CSF', 'PT181_AB42_CSF']
df_interest = final_df[final_df['METHOD']=='elecsys'][var_of_interest].copy(deep=True)
percentile_df = pd.DataFrame(index=df_interest.keys(), columns=['1° percentile',  '99° percentile', ])

for var in df_interest.keys():
    p1 = df_interest[var].quantile(0.01)
    p99 = df_interest[var].quantile(0.99)
    
    percentile_df.loc[var, '1° percentile'] = round(p1, 4)
    percentile_df.loc[var, '99° percentile'] = round(p99, 4)

print("1 e 99 percentili variabili CSF --> metodo ELECSYS")
print(percentile_df.dropna(how='all'))

## Single COFACTOR files

### APOERES

In [ ]:
file_code = 'APOERES'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['GENOTYPE'])


In [ ]:
print('prima', len(df_new),'\ndopo ', len(no_none_df))

In [ ]:
no_none_df.head()

In [ ]:
standardised_df = dataCleaner.uniform_APOE_format(df=no_none_df, col_name='GENOTYPE')
processed_df = dataCleaner.APOE_4_count(df=standardised_df, col_name='GENOTYPE')

In [ ]:
display(processed_df.head())

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['APOE_4'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
final_df

In [ ]:
# optaining automatically info to save the file
lst_population =  infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### PTDEMOG

In [ ]:
file_code = 'PTDEMOG'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)
processed_df['AGE_AD_BEG'] = processed_df.apply(lambda row: row['PTADBEG'] - row['PTDOBYY'], axis=1)
processed_df['AGE_AD_DX'] = processed_df.apply(lambda row: row['PTADDX'] - row['PTDOBYY'], axis=1)
processed_df['AGE_COG_BEG'] = processed_df.apply(lambda row: row['PTCOGBEG'] - row['PTDOBYY'], axis=1)

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['AGE', 'VISIT_MONTH', 'AGE_AD_BEG', 'AGE_AD_DX', 'AGE_COG_BEG'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### DXSUM

In [ ]:
file_code = 'DXSUM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
        }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='DIAGNOSIS')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### MMSE

In [ ]:
file_code = 'MMSE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['MMSCORE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADAS 13 & 11

In [ ]:
file_code = 'ADAS'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['TOTSCORE', 'TOTAL13']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### FAQ

In [ ]:
file_code = 'FAQ'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['FAQTOTAL']

no_unknow_df = dataCleaner.replace_unknown_values(df_new, new_nans=[-1])
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### CDR

In [ ]:
file_code = 'CDR'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['CDGLOBAL', 'CDRSB']

no_unknow_df = dataCleaner.replace_unknown_values(df_new, new_nans=[-1])
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
print(len(df_new))
print(len(no_none_df))

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
print(len(df_new))
print(len(final_df))

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### MOCA

In [ ]:
file_code = 'MOCA'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['MOCA']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [ ]:
print(len(df_new))
print(len(no_none_df))

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

In [ ]:
print(len(df_new))
print(len(final_df))

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
### KEEEP --> tentativo di calcolare MOCA dagli altri valori ma qualcosa non torna
'''
df_focus['VIS_SPACE'] = df_focus['TRAILS']+df_focus['CUBE']+df_focus['CLOCKCON']+df_focus['CLOCKNO']+df_focus['CLOCKHAN']
df_focus['NAMING']= df_focus['LION']+df_focus['RHINO']+df_focus['CAMEL']
df_focus['RECALL'] = df_focus['DELW1']+df_focus['DELW2']+df_focus['DELW3']+df_focus['DELW4']+df_focus['DELW5']

serial_sum = df_focus['SERIAL1']+df_focus['SERIAL2']+df_focus['SERIAL3']+df_focus['SERIAL4']+df_focus['SERIAL5']
serial_sum = serial_sum.map(lambda x: 0 if x <= 1 else 1 if x in [2,3] else 2 if x == 4 else 3 if x == 5 else print('serial_sum:', x, 'out of range'))
letter_score = df_focus['LETTERS'].map(lambda x: 1 if x <= 1 else 0 if x > 1 else print('letter_score:', x, 'out of range'))
df_focus['ATTENTION'] = serial_sum + letter_score + df_focus['DIGFOR'] + df_focus['DIGBACK']

fluency_score = df_focus['FFLUENCY'].map(lambda x: 1 if x>=11 else 0 if x<11 else print('fluency_score:', x, 'out of range'))
df_focus['LANGUAGE'] = fluency_score + df_focus['REPEAT1'] + df_focus['REPEAT2']

df_focus['ABSTRACTION'] = df_focus['ABSTRAN'] + df_focus['ABSMEAS']
df_focus['ORIENTATION'] = df_focus['DATE'] + df_focus['MONTH'] + df_focus['YEAR'] + df_focus['DAY']+df_focus['PLACE']+df_focus['CITY']

df_focus['MOCA_tot'] = df_focus['VIS_SPACE'] + df_focus['NAMING'] + df_focus['RECALL'] + df_focus['ATTENTION'] + df_focus['LANGUAGE']+ df_focus['ABSTRACTION'] + df_focus['ORIENTATION']
'''


## Not anymore used files

### YASSIN_CSF & YASSIN_PLASMA

In [ ]:
file_code = 'YASSINE_PLASMA'       #CSF
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['Phenotype'])


In [ ]:
print('prima', len(df_new),'\ndopo ', len(no_none_df))

In [ ]:
no_none_df.head()

In [ ]:
no_none_df.keys()

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= ['Phenotype'])

In [ ]:
standardised_df = dataCleaner.uniform_APOE_format(df=datefix_df, col_name='Phenotype')
processed_df = dataCleaner.APOE_4_count(df=standardised_df, col_name='Phenotype')

In [ ]:
display(processed_df.head())

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['APOE_4'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
final_df

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:

lst_population =  ['ADNI1', 'ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
new_file_name

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### BLCHANGE

In [ ]:
file_code = 'BLCHANGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_reference = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='BCPREDX')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)